In [2]:
# ==============================================================================
# MULTIVIEW LEAST SQUARES SUPPORT VECTOR MACHINE (MV-LSSVM) EXPERIMENT
# Compares 7 Subsampling Strategies:
# 1. MV-LSSVM: All samples
# 2. MV-LSSVM: Global Random (same samples removed across all views)
# 3. MV-LSSVM: Uniform Random (per view)
# 4. MV-LSSVM: Nyström (per view)
# 5. MV-LSSVM: Unsupervised clustering (m points per view)
# 6. MV-LSSVM: Stratified Kernel Herding (per view)
# 7. MV-LSSVM: Proposed Stratified HSIC Herding + Warmup CV (per view)
# ==============================================================================

import os
import time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.cluster import KMeans
from sklearn.datasets import load_digits
from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.preprocessing import OneHotEncoder

# --- 0. Output Directory Setup ---
OUTPUT_DIR = "8_mvlssvm_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"[INFO] Artifacts will be saved to: {os.path.abspath(OUTPUT_DIR)}")


# --- 1. MV-LSSVM Classifier Implementation ---

class Multiclass_MV_LSSVM(BaseEstimator, ClassifierMixin):
    """
    Multiclass Multi-View Least Squares Support Vector Machine (MV-LSSVM).
    Solves a coupled primal-dual linear system across multiple views.
    """
    def __init__(self, kernel_gammas, selected_indices, C=1.0):
        self.kernel_gammas = kernel_gammas
        self.selected_indices = selected_indices
        self.C = C

    def fit(self, X_list, y):
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)
        n_views = len(X_list)

        # One-hot encode targets to (-1, +1) matrices for LSSVM regression formulation
        ohe = OneHotEncoder(sparse_output=False)
        Y_onehot = ohe.fit_transform(y.reshape(-1, 1))
        Y_onehot = np.where(Y_onehot == 0, -1.0, 1.0)  # Shape: (N_train, K)

        self.sub_X_ = []
        self.sub_Y_ = []
        
        # Build coupled linear system blocks
        K_blocks = []
        total_m = 0
        
        for v in range(n_views):
            idx_v = np.array(self.selected_indices[v], dtype=int)
            X_v_sub = X_list[v][idx_v]
            Y_v_sub = Y_onehot[idx_v]
            
            self.sub_X_.append(X_v_sub)
            self.sub_Y_.append(Y_v_sub)
            
            m_v = len(idx_v)
            total_m += m_v
            
            # Compute RBF Kernel matrix for sub-samples of view v
            Kv = rbf_kernel(X_v_sub, gamma=self.kernel_gammas[v])
            K_blocks.append(Kv)

        # Assemble block-diagonal Kernel matrix K_tilde and regularization term
        K_tilde = np.zeros((total_m, total_m))
        Y_tilde = np.zeros((total_m, n_classes))
        
        current_idx = 0
        for v in range(n_views):
            m_v = len(self.selected_indices[v])
            sl = slice(current_idx, current_idx + m_v)
            K_tilde[sl, sl] = K_blocks[v]
            Y_tilde[sl, :] = self.sub_Y_[v]
            current_idx += m_v

        # Linear System: (K_tilde + (1/C) * I) Alpha = Y_tilde
        A = K_tilde + (1.0 / self.C) * np.eye(total_m)
        
        # Solve for dual variables (Alpha) across all class dimensions simultaneously
        self.alpha_ = np.linalg.solve(A, Y_tilde)
        return self

    def predict(self, X_list):
        n_samples = X_list[0].shape[0]
        n_classes = len(self.classes_)
        n_views = len(X_list)
        
        f_scores = np.zeros((n_samples, n_classes))
        
        current_idx = 0
        for v in range(n_views):
            m_v = len(self.selected_indices[v])
            sl = slice(current_idx, current_idx + m_v)
            
            # Evaluation kernel between test points and subsampled training points
            K_test_v = rbf_kernel(X_list[v], self.sub_X_[v], gamma=self.kernel_gammas[v])
            
            # Accumulate predictions from view v
            f_scores += K_test_v @ self.alpha_[sl, :]
            current_idx += m_v

        return self.classes_[np.argmax(f_scores, axis=1)]


# --- 2. Subsampling Strategies ---

def compute_centered_gram_matrix(X, gamma):
    K = rbf_kernel(X, gamma=gamma)
    n = K.shape[0]
    H = np.eye(n) - (1.0 / n) * np.ones((n, n))
    return H @ K @ H

def get_all_samples_indices(y_train, V):
    return [np.arange(len(y_train)) for _ in range(V)]

def get_global_random_indices(y_train, m, V, rng):
    """Removes randomly the exact same sample indices across all views."""
    shared_indices = rng.choice(len(y_train), size=m, replace=False)
    return [shared_indices.copy() for _ in range(V)]

def get_uniform_random_indices(y_train, m, V, rng):
    """Uniform random independently per view."""
    return [rng.choice(len(y_train), size=m, replace=False) for _ in range(V)]

def get_nystrom_indices(y_train, m, V, rng):
    return [rng.choice(len(y_train), size=m, replace=False) for _ in range(V)]

def get_kmeans_centroid_indices(views_train, m):
    selected = []
    for _, X_v in views_train.items():
        kmeans = KMeans(n_clusters=m, n_init=3, random_state=42).fit(X_v)
        closest_idx = []
        for c in kmeans.cluster_centers_:
            dists = np.linalg.norm(X_v - c, axis=1)
            for idx in np.argsort(dists):
                if idx not in closest_idx:
                    closest_idx.append(idx)
                    break
        selected.append(np.array(closest_idx, dtype=int))
    return selected

def get_stratified_kernel_herding_indices(views_train, y_train, gammas, m):
    """Stratified Classwise Kernel Herding per view."""
    classes = np.unique(y_train)
    m_per_class = max(1, m // len(classes))
    view_names = list(views_train.keys())
    selected_indices = {v_name: [] for v_name in view_names}

    for cls in classes:
        cls_mask = y_train == cls
        cls_indices = np.where(cls_mask)[0]
        n_cls = len(cls_indices)
        if n_cls == 0: continue

        for v_name in view_names:
            X_v_cls = views_train[v_name][cls_mask]
            K = rbf_kernel(X_v_cls, gamma=gammas[v_name])
            mu_k = np.mean(K, axis=1)
            chosen = []
            
            for t in range(min(m_per_class, n_cls)):
                if len(chosen) == 0:
                    next_idx = np.argmax(mu_k)
                else:
                    score = (t + 1) * mu_k - np.sum(K[:, chosen], axis=1)
                    score[chosen] = -np.inf
                    next_idx = np.argmax(score)
                chosen.append(next_idx)

            selected_indices[v_name].extend(cls_indices[chosen])

    return [np.array(selected_indices[v_name], dtype=int) for v_name in view_names]

def get_proposed_hsic_herding_indices(views_train, y_train, gammas, m, warmup_ratio=0.1, random_state=42):
    """Proposed Classwise Centered-Gram HSIC Herding with Warmup."""
    rng = np.random.default_rng(random_state)
    classes = np.unique(y_train)
    m_per_class = max(1, m // len(classes))
    w_per_class = int(m_per_class * warmup_ratio)

    view_names = list(views_train.keys())
    selected_indices = {v_name: [] for v_name in view_names}

    for cls in classes:
        cls_mask = y_train == cls
        cls_indices = np.where(cls_mask)[0]
        n_cls = len(cls_indices)
        if n_cls == 0: continue

        grams = [compute_centered_gram_matrix(views_train[v_name][cls_mask], gammas[v_name]) for v_name in view_names]

        for v_idx, v_name in enumerate(view_names):
            K = grams[v_idx]
            chosen = []
            if w_per_class > 0:
                chosen.extend(rng.choice(n_cls, size=min(w_per_class, n_cls), replace=False).tolist())

            mu_k = np.mean(K, axis=1)
            for t in range(len(chosen), min(m_per_class, n_cls)):
                if len(chosen) == 0:
                    next_idx = np.argmax(mu_k)
                else:
                    score = (t + 1) * mu_k - np.sum(K[:, chosen], axis=1)
                    score[chosen] = -np.inf
                    next_idx = np.argmax(score)
                chosen.append(next_idx)

            selected_indices[v_name].extend(cls_indices[chosen])

    return [np.array(selected_indices[v_name], dtype=int) for v_name in view_names]


# --- 3. Hyperparameter Tuning ---

def tune_hsic_warmup_ratio(views_train, y_train, gammas_list, m, candidate_alphas=[0.0, 0.05, 0.1, 0.2, 0.3], cv=3):
    """3-Fold Inner Cross-Validation for HSIC Herding Warmup Ratio (alpha)."""
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
    view_names = list(views_train.keys())
    best_alpha = candidate_alphas[0]
    best_score = -1.0

    for alpha in candidate_alphas:
        cv_scores = []
        for train_fold_idx, val_fold_idx in skf.split(views_train[view_names[0]], y_train):
            v_tr_fold = {v: views_train[v][train_fold_idx] for v in view_names}
            v_val_fold = [views_train[v][val_fold_idx] for v in view_names]
            y_tr_fold, y_val_fold = y_train[train_fold_idx], y_train[val_fold_idx]

            sub_idx = get_proposed_hsic_herding_indices(
                v_tr_fold, y_tr_fold, dict(zip(view_names, gammas_list)), m, warmup_ratio=alpha, random_state=42
            )

            clf = Multiclass_MV_LSSVM(kernel_gammas=gammas_list, selected_indices=sub_idx)
            clf.fit([v_tr_fold[v] for v in view_names], y_tr_fold)
            preds = clf.predict(v_val_fold)
            cv_scores.append(accuracy_score(y_val_fold, preds))

        mean_acc = np.mean(cv_scores)
        if mean_acc > best_score:
            best_score = mean_acc
            best_alpha = alpha

    return best_alpha


# --- 4. Dataset Loading & Setup ---

print("\n>>> Loading 10-Class Digits dataset for MV-LSSVM...")
digits = load_digits(n_class=10)
X_all, y_all = digits.data, digits.target

train_idx, test_idx = train_test_split(np.arange(len(y_all)), test_size=0.3, stratify=y_all, random_state=42)
X_images = X_all.reshape(-1, 8, 8)

views_tr = {f"Row_{r+1}": X_images[train_idx, r, :] for r in range(8)}
views_te = {f"Row_{r+1}": X_images[test_idx, r, :] for r in range(8)}
y_tr, y_te = y_all[train_idx], y_all[test_idx]

view_names = list(views_tr.keys())
V = len(view_names)
X_tr_list = [views_tr[v] for v in view_names]
X_te_list = [views_te[v] for v in view_names]

# Simple Gamma estimation per view
gammas_dict = {v_name: 1.0 / (views_tr[v_name].shape[1] * views_tr[v_name].var()) for v_name in view_names}
gammas_list = [gammas_dict[v] for v in view_names]


# ==============================================================================
# SECTION A: MV-LSSVM EXPERIMENT RUNNER
# ==============================================================================
print("\n>>> Running MV-LSSVM Benchmark across Subsample Budgets...")

subsample_sizes = [10, 20, 30, 40, 50]
n_runs = 3

strategies = [
    "MV-LSSVM: All samples",
    "MV-LSSVM: Global Random (Same samples removed)",
    "MV-LSSVM: Uniform Random (Per view)",
    "MV-LSSVM: Nyström (Per view)",
    "MV-LSSVM: Unsupervised clustering",
    "MV-LSSVM: Stratified Kernel Herding",
    "MV-LSSVM: Proposed Stratified HSIC Herding"
]

results = {s: {m: [] for m in subsample_sizes} for s in strategies}
fit_times = {s: {m: [] for m in subsample_sizes} for s in strategies}
selected_alphas = {m: [] for m in subsample_sizes}

for m in subsample_sizes:
    # Inner CV: Tune Warmup Ratio (Alpha) specifically for MV-LSSVM at budget m
    best_alpha = tune_hsic_warmup_ratio(views_tr, y_tr, gammas_list, m, candidate_alphas=[0.0, 0.05, 0.1, 0.2, 0.3])
    selected_alphas[m] = best_alpha
    print(f"    [Budget m={m:02d}] MV-LSSVM Tuned HSIC Warmup Ratio (Alpha): {best_alpha}")

    for run in range(n_runs):
        rng = np.random.default_rng(run)
        
        sub_samples = {
            "MV-LSSVM: All samples": get_all_samples_indices(y_tr, V),
            "MV-LSSVM: Global Random (Same samples removed)": get_global_random_indices(y_tr, m, V, rng),
            "MV-LSSVM: Uniform Random (Per view)": get_uniform_random_indices(y_tr, m, V, rng),
            "MV-LSSVM: Nyström (Per view)": get_nystrom_indices(y_tr, m, V, rng),
            "MV-LSSVM: Unsupervised clustering": get_kmeans_centroid_indices(views_tr, m),
            "MV-LSSVM: Stratified Kernel Herding": get_stratified_kernel_herding_indices(views_tr, y_tr, gammas_dict, m),
            "MV-LSSVM: Proposed Stratified HSIC Herding": get_proposed_hsic_herding_indices(
                views_tr, y_tr, gammas_dict, m, warmup_ratio=best_alpha, random_state=run
            )
        }

        for strat in strategies:
            t0 = time.time()
            clf = Multiclass_MV_LSSVM(kernel_gammas=gammas_list, selected_indices=sub_samples[strat])
            clf.fit(X_tr_list, y_tr)
            preds = clf.predict(X_te_list)
            elapsed = time.time() - t0

            results[strat][m].append(1.0 - accuracy_score(y_te, preds))
            fit_times[strat][m].append(elapsed)

# Save CSV Results
csv_rows = []
for strat in strategies:
    for m in subsample_sizes:
        csv_rows.append({
            "strategy": strat,
            "subsample_size_m": m,
            "tuned_alpha": selected_alphas[m] if "Proposed" in strat else np.nan,
            "mean_error_rate": np.mean(results[strat][m]),
            "std_error_rate": np.std(results[strat][m]),
            "mean_execution_time_sec": np.mean(fit_times[strat][m]),
            "std_execution_time_sec": np.std(fit_times[strat][m])
        })

df_results = pd.DataFrame(csv_rows)
df_results.to_csv(os.path.join(OUTPUT_DIR, "mvlssvm_results.csv"), index=False)


# ==============================================================================
# SECTION B: VISUALIZATIONS
# ==============================================================================

styles = {
    "MV-LSSVM: All samples": ("--", "black", 2.0),
    "MV-LSSVM: Global Random (Same samples removed)": (":", "gray", 1.5),
    "MV-LSSVM: Uniform Random (Per view)": ("-.", "orange", 1.5),
    "MV-LSSVM: Nyström (Per view)": (":", "blue", 1.5),
    "MV-LSSVM: Unsupervised clustering": ("--", "purple", 1.5),
    "MV-LSSVM: Stratified Kernel Herding": ("-.", "green", 1.8),
    "MV-LSSVM: Proposed Stratified HSIC Herding": ("-", "red", 2.5)
}

# --- Plot 1: Error Rate ---
plt.figure(figsize=(10, 5.5))
for strat in strategies:
    means = [np.mean(results[strat][m]) for m in subsample_sizes]
    ls, col, lw = styles[strat]
    plt.plot(subsample_sizes, means, linestyle=ls, color=col, linewidth=lw, marker="o", label=strat)

plt.title("MV-LSSVM Performance: Subsample Size vs. Error Rate")
plt.xlabel("Subsample Size per View ($m$)")
plt.ylabel("Test Classification Error Rate")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend(bbox_to_anchor=(1.04, 1), loc="upper left")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "mvlssvm_error.png"), dpi=300)
plt.close()

# --- Plot 2: Runtime ---
plt.figure(figsize=(10, 5.5))
for strat in strategies:
    t_means = [np.mean(fit_times[strat][m]) for m in subsample_sizes]
    ls, col, lw = styles[strat]
    plt.plot(subsample_sizes, t_means, linestyle=ls, color=col, linewidth=lw, marker="s", label=strat)

plt.title("MV-LSSVM Computational Efficiency: Execution Time")
plt.xlabel("Subsample Size per View ($m$)")
plt.ylabel("Execution Time (Seconds)")
plt.yscale("log")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend(bbox_to_anchor=(1.04, 1), loc="upper left")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "mvlssvm_runtime.png"), dpi=300)
plt.close()


# ==============================================================================
# SECTION C: SUMMARY TABLE
# ==============================================================================
print("\n" + "=" * 90)
print(f"{'MV-LSSVM Strategy':<42} | {'m=10 Error':<10} | {'m=50 Time (s)':<12} | {'m=50 Error':<10} | {'Alpha':<8}")
print("=" * 90)
for strat in strategies:
    e10 = np.mean(results[strat][10])
    e50 = np.mean(results[strat][50])
    t50 = np.mean(fit_times[strat][50])
    a_str = f"{selected_alphas[50]:.2f}" if "Proposed" in strat else "N/A"
    print(f"{strat:<42} | {e10:<10.4f} | {t50:<12.4f} | {e50:<10.4f} | {a_str:<8}")
print("=" * 90)

print(f"\n[SUCCESS] MV-LSSVM experiments complete. Output saved to '{OUTPUT_DIR}'")

[INFO] Artifacts will be saved to: /Users/blancacanocamarero/tesis/repositories/subsampling-for-multi-view-kernel-methods/0_preliminar/MVLSVM/8_mvlssvm_results

>>> Loading 10-Class Digits dataset for MV-LSSVM...

>>> Running MV-LSSVM Benchmark across Subsample Budgets...
    [Budget m=10] MV-LSSVM Tuned HSIC Warmup Ratio (Alpha): 0.0
    [Budget m=20] MV-LSSVM Tuned HSIC Warmup Ratio (Alpha): 0.0
    [Budget m=30] MV-LSSVM Tuned HSIC Warmup Ratio (Alpha): 0.0
    [Budget m=40] MV-LSSVM Tuned HSIC Warmup Ratio (Alpha): 0.3
    [Budget m=50] MV-LSSVM Tuned HSIC Warmup Ratio (Alpha): 0.2

MV-LSSVM Strategy                          | m=10 Error | m=50 Time (s) | m=50 Error | Alpha   
MV-LSSVM: All samples                      | 0.0796     | 4.4014       | 0.0796     | N/A     
MV-LSSVM: Global Random (Same samples removed) | 0.6426     | 0.0101       | 0.2716     | N/A     
MV-LSSVM: Uniform Random (Per view)        | 0.4549     | 0.0080       | 0.1821     | N/A     
MV-LSSVM: Nyström (Pe